In [ ]:
# 1️ IMPORT REQUIRED LIBRARIES

import os
import cv2
import pandas as pd
import pydicom
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

In [ ]:
# 2️ DICOM TO PNG CONVERSION FUNCTION


def dicom_to_png(dicom_path, png_path):
    """
    Converts a DICOM medical image to PNG format.
    """
    dcom = pydicom.dcmread(dicom_path)
    img = dcom.pixel_array  # FIXED (previous typo)
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
    cv2.imwrite(png_path, img)

In [ ]:
# 3️ CUSTOM DATASET CLASS

class CSRClassificationDataset(Dataset):
    """
    Custom dataset for loading PNG chest X-ray images.
    """
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)  # FIXED BUG

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# 4️ DEVICE CONFIGURATION


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# 5️ IMAGE TRANSFORMATIONS

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:

# 6 DATASET PATH & LOADING (ADD BELOW TRANSFORM)

train_dir = "chest_xray/train"
val_dir   = "chest_xray/test"

from torchvision import datasets

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Classes detected:", train_dataset.classes)

In [ ]:
# 7 MODEL DEFINITION (YOUR ORIGINAL STYLE - CLEANED)

def build_resnet():
    """
    Builds ResNet50 with custom classification head.
    Binary output (1 neuron).
    """
    model = models.resnet50(pretrained=True)

    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 1)  # Single output for binary classification
        # NOTE: No Sigmoid here because we use BCEWithLogitsLoss
    )

    return model

model = build_resnet().to(device)

In [ ]:
# 8 LOSS FUNCTION & OPTIMIZER

criterion = nn.BCEWithLogitsLoss()  # Correct for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 9 TRAINING LOOP

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.unsqueeze(1).to(device)  # Shape fix for BCE

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss:.4f}")



In [ ]:
# 10 SAVE MODEL

torch.save(model.state_dict(), "model.pth")
print("Model saved successfully!")